# Final Project STI - Kelompok 5
## Analisis Kinerja Penjualan Ritel Online (2023-2026)
**Pendekatan: CRISP-DM** (Business -> Data Understanding -> Data Preparation -> Modeling/Analisis -> Evaluation) | **Output: dashboard interaktif (Plotly) langsung di notebook**

> Studi kasus: data penjualan B2C sebuah retailer online (data agregat mingguan, anonim - tanpa identitas pelanggan).

## 1. Business Understanding

**Konteks.** Sebuah retailer online B2C ingin memahami kinerja penjualannya selama ~3,5 tahun untuk mendukung keputusan operasional (stok, promo, target).

**Business questions:**
1. Bagaimana **tren & pertumbuhan** penjualan (revenue, transaksi, pelanggan) dari waktu ke waktu?
2. Apa **pendorong** pertumbuhan revenue - kenaikan jumlah transaksi (volume) atau nilai per transaksi (AOV)?
3. Apakah ada **pola musiman** (bulan ramai/sepi) untuk perencanaan stok & campaign?
4. Seberapa **efektif strategi diskon** terhadap revenue?

**Output.** Dashboard monitoring **interaktif (Plotly)** langsung di dalam notebook ini (hover, zoom, filter).

## 2. Data Understanding

In [ ]:
import pandas as pd
import numpy as np
import io
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

# Data penjualan mingguan ditanam langsung agar notebook portabel
# (tidak perlu mount Google Drive / file eksternal).
DATA = """week,trx,qty,avg_price,disc_share,cust,revenue
2022-12-26,17,17,690064,0.1176,16,4245000
2023-01-02,270,276,391171,0.1556,265,94677530
2023-01-09,193,208,216861,0.1554,188,58215910
2023-01-16,311,339,410754,0.2637,280,99222409
2023-01-23,360,380,191869,0.3056,323,82278010
2023-01-30,405,436,267776,0.2765,358,100269160
2023-02-06,366,383,359021,0.2623,312,87590670
2023-02-13,340,472,244302,0.2176,305,145672530
2023-02-20,244,262,274166,0.3238,213,66933780
2023-02-27,361,422,229282,0.2133,327,101869510
2023-03-06,332,365,364949,0.3404,291,92045960
2023-03-13,322,338,243996,0.3665,293,78003020
2023-03-20,274,312,219557,0.281,234,74603640
2023-03-27,323,351,262083,0.3653,282,92909929
2023-04-03,298,319,345240,0.3188,256,89149440
2023-04-10,323,361,264075,0.2941,271,113843500
2023-04-17,223,272,198741,0.2422,194,60071750
2023-04-24,162,183,216699,0.1605,150,59384520
2023-05-01,143,150,225648,0.1399,139,45584270
2023-05-08,164,202,252029,0.1341,156,60542270
2023-05-15,145,165,226876,0.1862,141,36760400
2023-05-22,188,199,294554,0.2181,171,62361480
2023-05-29,277,294,317596,0.3863,239,75263380
2023-06-05,265,288,228076,0.3283,234,87185710
2023-06-12,242,287,272964,0.3347,222,75780780
2023-06-19,276,318,289596,0.3732,243,67634760
2023-06-26,187,196,243571,0.3957,178,43655625
2023-07-03,245,265,190212,0.4082,222,60334880
2023-07-10,141,179,208337,0.461,128,46106500
2023-07-17,163,171,315858,0.5583,146,49680627
2023-07-24,104,116,277710,0.5769,94,32800981
2023-07-31,185,190,189657,0.3459,169,41191600
2023-08-07,191,201,234869,0.2932,177,47485630
2023-08-14,252,282,220428,0.3135,232,66373500
2023-08-21,250,264,228015,0.336,231,64505020
2023-08-28,270,312,278659,0.3259,236,96611239
2023-09-04,227,241,177797,0.2819,211,51050660
2023-09-11,224,246,251886,0.3036,202,54876850
2023-09-18,239,298,228120,0.3138,204,71499280
2023-09-25,199,227,247382,0.2915,182,53906280
2023-10-02,240,265,410391,0.3583,205,72454500
2023-10-09,205,219,218542,0.322,183,55923890
2023-10-16,175,186,238424,0.4343,148,46927890
2023-10-23,191,213,207472,0.3717,174,53910630
2023-10-30,207,230,204674,0.3527,190,55981160
2023-11-06,202,231,269999,0.3564,176,73640098
2023-11-13,211,228,473188,0.4265,178,90789140
2023-11-20,119,217,161684,0.4874,92,72824300
2023-11-27,171,222,179039,0.4737,149,68835664
2023-12-04,181,193,282542,0.3978,171,65233589
2023-12-11,217,270,417793,0.4378,197,131980508
2023-12-18,199,216,285425,0.4623,192,63896940
2023-12-25,271,311,390265,0.4428,175,132040910
2024-01-01,315,340,349830,0.4317,205,110670568
2024-01-08,390,427,293102,0.4846,233,121838530
2024-01-15,370,385,321773,0.373,221,114125490
2024-01-22,312,349,289283,0.4455,186,94884140
2024-01-29,278,325,305322,0.4101,177,113494800
2024-02-05,278,300,304339,0.4928,173,89507592
2024-02-12,300,317,339266,0.4567,179,97666280
2024-02-19,276,355,235371,0.4167,251,129758635
2024-02-26,457,581,258507,0.5361,286,162579862
2024-03-04,398,426,369690,0.495,269,139299434
2024-03-11,345,447,262134,0.3942,216,168217170
2024-03-18,362,619,171443,0.453,216,207003510
2024-03-25,383,427,229969,0.4491,249,103122270
2024-04-01,399,421,323131,0.4687,234,96592626
2024-04-08,143,143,181434,0.2937,111,34948730
2024-04-15,321,426,267394,0.4455,201,160953050
2024-04-22,357,427,312192,0.4342,204,132933156
2024-04-29,336,392,294020,0.4405,221,163830890
2024-05-06,326,361,329778,0.4693,197,102290450
2024-05-13,379,436,319737,0.4828,221,139607690
2024-05-20,303,335,308247,0.4851,177,119863500
2024-05-27,372,395,360403,0.4866,223,152387242
2024-06-03,361,457,339281,0.4792,213,208129636
2024-06-10,382,424,354801,0.4503,221,178605156
2024-06-17,255,292,293418,0.5059,158,105436642
2024-06-24,365,469,258354,0.4877,219,113360370
2024-07-01,386,560,312444,0.5052,232,223721301
2024-07-08,339,374,390895,0.4808,197,105737750
2024-07-15,554,617,326240,0.6318,309,183869740
2024-07-22,523,601,340028,0.608,297,222885618
2024-07-29,487,570,311964,0.5236,271,195452410
2024-08-05,426,505,268829,0.5164,230,141192434
2024-08-12,350,391,390016,0.4886,198,171815300
2024-08-19,295,363,298773,0.4678,166,129832000
2024-08-26,357,402,291049,0.4454,189,121410054
2024-09-02,335,387,469043,0.4657,164,222890710
2024-09-09,340,356,539595,0.4382,180,142743728
2024-09-16,312,327,363758,0.4744,179,114240890
2024-09-23,312,354,377435,0.4038,230,137738000
2024-09-30,324,349,398796,0.3611,287,114955640
2024-10-07,246,264,488445,0.3496,211,83907780
2024-10-14,149,155,449360,0.3087,141,39936170
2024-10-21,216,241,342836,0.287,175,55632778
2024-10-28,514,635,487717,0.3171,301,216433330
2024-11-04,539,635,377221,0.2857,315,276292022
2024-11-11,502,554,570321,0.2749,282,312400400
2024-11-18,409,543,382426,0.2958,223,237348142
2024-11-25,391,441,455479,0.3402,238,175879753
2024-12-02,444,499,590385,0.2748,257,290567957
2024-12-09,425,500,408123,0.3506,251,222536220
2024-12-16,393,433,463375,0.2926,239,127729500
2024-12-23,416,475,313529,0.238,214,177989600
2024-12-30,407,565,274989,0.2432,187,178059975
2025-01-06,593,869,307709,0.3845,228,421893799
2025-01-13,572,738,360189,0.3759,219,391766340
2025-01-20,548,612,481485,0.3339,225,256774905
2025-01-27,251,271,359669,0.4024,170,82543200
2025-02-03,328,390,395685,0.3659,237,159438391
2025-02-10,344,419,551683,0.4099,242,212013485
2025-02-17,377,503,290230,0.4191,262,164744366
2025-02-24,506,841,205807,0.4071,231,535679742
2025-03-03,667,902,240481,0.3718,255,397330009
2025-03-10,600,632,402559,0.3083,228,214141416
2025-03-17,701,1026,323926,0.2996,282,414072619
2025-03-24,541,853,200786,0.3919,213,260205415
2025-03-31,280,325,332315,0.3107,120,86642721
2025-04-07,613,745,255897,0.3214,220,203044271
2025-04-14,499,711,400910,0.2886,187,526107667
2025-04-21,466,565,353725,0.3476,192,258662784
2025-04-28,372,417,271983,0.3548,181,143043816
2025-05-05,374,448,255088,0.2914,200,167698414
2025-05-12,354,478,426097,0.2712,198,151543938
2025-05-19,442,541,231280,0.2851,206,206465448
2025-05-26,515,579,313435,0.3476,197,222612239
2025-06-02,525,551,644041,0.3524,193,332453143
2025-06-09,568,679,365116,0.3398,212,329105905
2025-06-16,557,1776,205883,0.3573,207,1075472646
2025-06-23,478,549,526976,0.3368,175,219457390
2025-06-30,583,650,105393244,0.3619,227,209905848
2025-07-07,565,923,327612,0.315,217,1136220013
2025-07-14,637,959,393370,0.314,224,578602669
2025-07-21,629,758,317007,0.3386,227,444397701
2025-07-28,631,781,346148,0.3407,237,1327198296
2025-08-04,598,686,402378,0.2659,225,297206493
2025-08-11,452,567,319776,0.3186,169,194864510
2025-08-18,496,2197,115827,0.375,190,1128962682
2025-08-25,360,474,144429314,0.3944,182,246532474
2025-09-01,392,510,380789,0.4566,204,174620414
2025-09-08,380,498,440727,0.4684,211,184077276
2025-09-15,363,633,108130733,0.4408,197,484565980
2025-09-22,315,520,294916,0.4254,239,219437319
2025-09-29,437,579,269538,0.3249,344,264070854
2025-10-06,473,628,261274,0.408,378,263447073
2025-10-13,341,398,323145,0.4927,273,192638891
2025-10-20,410,497,327058,0.4463,314,269286940
2025-10-27,367,434,267386,0.4605,296,129594607
2025-11-03,353,406,421245,0.4051,305,241011836
2025-11-10,340,472,248580,0.4176,289,196997106
2025-11-17,311,513,177859,0.3987,236,166183482
2025-11-24,280,1344,60861,0.3964,225,643045945
2025-12-01,345,1236,99280,0.4058,265,588651307
2025-12-08,330,774,183702,0.3515,250,414763793
2025-12-15,463,588,224219,0.3305,313,223261833
2025-12-22,346,399,171741129,0.3295,253,519263575
2025-12-29,284,341,222192,0.3451,224,112943364
2026-01-05,386,1938,70023,0.4119,299,597298516
2026-01-12,328,634,623999340,0.4146,287,358892149
2026-01-19,339,726,1183676795,0.2301,284,301903250
2026-01-26,363,1386,117780,0.135,267,569881934
2026-02-02,350,381,266332,0.2571,271,180786450
2026-02-09,348,382,264031,0.2213,275,175974107
2026-02-16,287,318,267316,0.2787,239,117732168
2026-02-23,369,390,330398,0.3523,292,153586351
2026-03-02,375,403,354382,0.2827,305,182721774
2026-03-09,336,589,247052,0.2381,288,325085798
2026-03-16,231,257,290765,0.3463,205,82670026
2026-03-23,284,299,397680,0.338,250,112164399
2026-03-30,256,263,1055419,0.3516,223,127709287
2026-04-06,293,629,448095,0.3584,256,272961234
2026-04-13,310,389,775276,0.3871,271,171427318
2026-04-20,175,190,427042,0.36,162,90681080
2026-05-11,1,1,3090919,0.0,1,3090919"""

df = pd.read_csv(io.StringIO(DATA))
df['week'] = pd.to_datetime(df['week'], errors='coerce')
df = df.dropna(subset=['week']).sort_values('week').reset_index(drop=True)
print('Dimensi:', df.shape)
print('Periode:', df['week'].min().date(), '->', df['week'].max().date())
df.head()

**Kamus data (agregat per minggu):** `week` (tanggal awal minggu), `trx` (jumlah transaksi), `qty` (unit terjual), `avg_price` (harga rata-rata), `disc_share` (porsi transaksi/nilai berdiskon, 0-1), `cust` (jumlah pelanggan), `revenue` (pendapatan).

In [ ]:
df.info()
df.describe().T

In [ ]:
# Kualitas data: nilai hilang
df.isnull().sum()

## 3. Data Preparation
Menurunkan kolom waktu & metrik turunan (KPI) untuk analisis dan dashboard.

In [ ]:
df['year']          = df['week'].dt.year
df['month']         = df['week'].dt.month
df['month_name']    = df['week'].dt.strftime('%b')
df['quarter']       = 'Q' + df['week'].dt.quarter.astype(str)
df['yearmonth']     = df['week'].dt.strftime('%Y-%m')

df['aov']           = (df['revenue'] / df['trx']).round(0)     # Average Order Value
df['units_per_trx'] = (df['qty'] / df['trx']).round(2)         # ukuran keranjang
df['rev_per_cust']  = (df['revenue'] / df['cust']).round(0)
df['disc_pct']      = (df['disc_share'] * 100).round(1)
df['rev_4w_ma']     = df['revenue'].rolling(4, min_periods=1).mean().round(0)  # smoothing

# tandai tahun parsial (data tidak penuh 1 tahun) supaya tidak salah baca YoY
weeks_per_year = df.groupby('year')['week'].count()
df['full_year'] = df['year'].map(weeks_per_year >= 50)
df[['week','revenue','trx','aov','units_per_trx','year','full_year']].head()

## 4. Analisis & Dashboard Interaktif

> Grafik di bawah **interaktif** (Plotly): arahkan kursor untuk detail (hover), drag untuk zoom, pakai **range slider** & **dropdown tahun** pada grafik tren.

In [ ]:
# KPI ringkas (untuk scorecard dashboard)
print(f"Total revenue      : {df['revenue'].sum():,.0f}")
print(f"Total transaksi    : {df['trx'].sum():,.0f}")
print(f"Total unit         : {df['qty'].sum():,.0f}")
print(f"AOV keseluruhan    : {df['revenue'].sum()/df['trx'].sum():,.0f}")
print(f"Rata2 unit/transaksi: {df['units_per_trx'].mean():.2f}")
print(f"Rata2 porsi diskon : {df['disc_share'].mean()*100:.1f}%")

In [ ]:
# === DASHBOARD INTERAKTIF (Plotly) - KPI cards ===
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio
pio.renderers.default = 'colab'   # render interaktif di Colab

INK='#1C6F8C'; LT='#9ecae1'; ACC='#D98A4B'; GRN='#2E8B6F'; BL='#5FA8BE'
tot_rev = df['revenue'].sum(); tot_trx = df['trx'].sum(); aov_all = tot_rev/tot_trx

kpi = go.Figure()
kpi.add_trace(go.Indicator(mode='number', value=tot_rev, title={'text':'Total Revenue'},
    number={'prefix':'Rp ','valueformat':',.0f'}, domain={'row':0,'column':0}))
kpi.add_trace(go.Indicator(mode='number', value=tot_trx, title={'text':'Total Transaksi'},
    number={'valueformat':',.0f'}, domain={'row':0,'column':1}))
kpi.add_trace(go.Indicator(mode='number', value=aov_all, title={'text':'AOV (Rp)'},
    number={'valueformat':',.0f'}, domain={'row':0,'column':2}))
kpi.add_trace(go.Indicator(mode='number', value=df['disc_share'].mean()*100,
    title={'text':'Rata-rata Diskon'}, number={'suffix':' %','valueformat':'.1f'},
    domain={'row':0,'column':3}))
kpi.update_layout(grid={'rows':1,'columns':4,'pattern':'independent'}, height=170,
    margin=dict(t=50,b=10), title_text='<b>Ringkasan KPI - Penjualan Ritel Online</b>')
kpi.show()

In [ ]:
# Q1/Q2 - Ringkasan per tahun + pertumbuhan YoY (hanya tahun penuh utk YoY)
by_year = df.groupby('year').agg(
    weeks=('week','count'), revenue=('revenue','sum'),
    trx=('trx','sum'), cust=('cust','sum')).copy()
by_year['aov'] = (by_year['revenue']/by_year['trx']).round(0)
full = by_year[by_year['weeks']>=50].copy()
full['yoy_revenue_%'] = (full['revenue'].pct_change()*100).round(1)
full['yoy_aov_%']     = (full['aov'].pct_change()*100).round(1)
print('Catatan: tahun dengan <50 minggu (parsial) dikecualikan dari YoY.')
display(by_year)
display(full[['revenue','trx','aov','yoy_revenue_%','yoy_aov_%']])

In [ ]:
# Tren revenue mingguan (interaktif: hover, zoom, range slider, dropdown tahun)
trend = go.Figure()
trend.add_trace(go.Scatter(x=df['week'], y=df['revenue'], mode='lines', name='Mingguan',
    line=dict(color=LT,width=1), hovertemplate='%{x|%d %b %Y}<br>Rp %{y:,.0f}<extra></extra>'))
trend.add_trace(go.Scatter(x=df['week'], y=df['rev_4w_ma'], mode='lines',
    name='Rata-rata 4 minggu', line=dict(color=INK,width=2.5),
    hovertemplate='Rp %{y:,.0f}<extra></extra>'))
btns=[dict(label='Semua', method='relayout',
           args=[{'xaxis.range':[df['week'].min(), df['week'].max()]}])]
for yy in sorted(df['year'].unique()):
    w=df[df['year']==yy]['week']
    if len(w): btns.append(dict(label=str(yy), method='relayout',
        args=[{'xaxis.range':[w.min(), w.max()]}]))
trend.update_layout(title='<b>Tren Revenue Mingguan</b>', yaxis_title='Revenue (Rp)',
    hovermode='x unified', height=440, template='plotly_white',
    updatemenus=[dict(buttons=btns, x=1.0, y=1.18, xanchor='right', showactive=True)])
trend.update_xaxes(rangeslider_visible=True)
trend.show()

In [ ]:
# Pertumbuhan per tahun & pola musiman
order=['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']
ry=df.groupby('year')['revenue'].sum()
seas=df.groupby('month_name')['revenue'].mean().reindex(order)
fig=make_subplots(rows=1, cols=2,
    subplot_titles=('Revenue per Tahun','Rata-rata Revenue per Bulan (Seasonality)'))
fig.add_trace(go.Bar(x=ry.index.astype(str), y=ry.values, marker_color=INK,
    text=[f'{v/1e9:.1f} M' for v in ry.values], textposition='outside',
    hovertemplate='%{x}<br>Rp %{y:,.0f}<extra></extra>'),1,1)
fig.add_trace(go.Bar(x=seas.index, y=seas.values, marker_color=BL,
    hovertemplate='%{x}<br>Rp %{y:,.0f}<extra></extra>'),1,2)
fig.update_layout(height=400, showlegend=False, template='plotly_white',
    title_text='<b>Pertumbuhan & Pola Musiman</b>')
fig.show()

In [ ]:
# AOV per tahun & efektivitas diskon
ay=df.groupby('year')['aov'].mean()
fig=make_subplots(rows=1, cols=2, subplot_titles=('AOV per Tahun','Diskon (%) vs Revenue'))
fig.add_trace(go.Bar(x=ay.index.astype(str), y=ay.values, marker_color=GRN,
    text=[f'{v/1e3:.0f} rb' for v in ay.values], textposition='outside',
    hovertemplate='%{x}<br>AOV Rp %{y:,.0f}<extra></extra>'),1,1)
fig.add_trace(go.Scatter(x=df['disc_pct'], y=df['revenue'], mode='markers',
    marker=dict(color=ACC,opacity=0.6,size=7),
    hovertemplate='Diskon %{x:.0f}%<br>Rp %{y:,.0f}<extra></extra>'),1,2)
fig.update_xaxes(title_text='Porsi diskon (%)', row=1,col=2)
fig.update_yaxes(title_text='Revenue (Rp)', row=1,col=2)
corr=df['disc_share'].corr(df['revenue'])
fig.update_layout(height=400, showlegend=False, template='plotly_white',
    title_text=f'<b>AOV & Efektivitas Diskon</b>  (korelasi diskon-revenue {corr:.2f}, mendekati 0)')
fig.show()

## 5. Evaluation

**Temuan (lihat angka & chart di atas):**
- **Pertumbuhan kuat.** Revenue tahunan naik tajam: ~3,8 M (2023) -> ~8,0 M (2024, +112%) -> ~18,3 M (2025, +129%) - sekitar **5x dalam dua tahun**. (Catatan: 2022 hanya 1 minggu & 2026 baru ~17 minggu, jadi dikecualikan dari YoY.)
- **Pendorong = nilai, bukan sekadar volume.** AOV naik 307 rb (2023) -> 417 rb (2024) -> 776 rb (2025), sementara jumlah transaksi naik lebih moderat. Pertumbuhan revenue **lebih banyak ditarik kenaikan AOV** daripada jumlah transaksi.
- **Keranjang kecil.** Rata-rata ~1,3 unit/transaksi -> mayoritas beli 1 item -> ada peluang **cross-sell/bundling**.
- **Musiman.** Revenue cenderung **puncak di Juli** (juga tinggi Juni & Januari), lemah di Mei & Oktober -> acuan perencanaan stok & campaign.
- **Diskon belum efektif.** Rata-rata porsi diskon **~36% (tinggi)**, tetapi korelasi diskon vs revenue ~0 dan vs AOV justru negatif -> indikasi **over-discounting**: diskon besar tidak terbukti mengangkat revenue.

**Rekomendasi:**
1. Pertahankan momentum **AOV** (bundling/upsell) - keranjang masih 1,3 item.
2. **Evaluasi strategi diskon** (uji kurangi diskon di segmen tertentu, ukur dampak ke revenue).
3. Manfaatkan **puncak musiman** (Jun-Jul, Jan) untuk stok & promo.
4. Dorong **akuisisi/retensi pelanggan** (jumlah pelanggan relatif stagnan dibanding lonjakan revenue).

**Keterbatasan.** Data agregat mingguan tanpa dimensi produk/pelanggan/channel; periode 2026 masih berjalan (parsial).

## 6. Export Data (opsional)

In [ ]:
OUT = 'online_retail_weekly_enriched.csv'
df.to_csv(OUT, index=False)
print('Saved ke /content/' + OUT)
print('Data bersih + kolom turunan, kalau mau dipakai lagi di tool lain.')
# from google.colab import files; files.download(OUT)

### Rancangan Dashboard Looker Studio (saran)
**Sumber data:** `online_retail_weekly_enriched.csv` (sudah ada kolom turunan year/month/quarter/aov/dll).

1. **Scorecard (baris atas):** Total Revenue, Total Transaksi, AOV (rata-rata), Total Pelanggan, Rata-rata Diskon %.
2. **Time series:** Revenue per minggu/bulan (pakai `week`/`yearmonth`).
3. **Bar:** Revenue per Tahun (`year`) + Revenue per Bulan (`month_name`, seasonality).
4. **Combo/line:** AOV per bulan (tren nilai transaksi).
5. **Scatter:** `disc_pct` vs `revenue` (efektivitas diskon).
6. **Filter controls:** Tahun (`year`), Kuartal (`quarter`), rentang tanggal (`week`).